# Universal File Optimizer #

### Install Required Dependencies ###

In [1]:
print("Installing required libraries...")

!apt-get -qq install ghostscript -y

!pip -q install \
    pymupdf \
    pypdf \
    pillow \
    python-docx \
    openpyxl \
    tqdm

import os
import io
import json
import shutil
import zipfile
import tempfile
import subprocess
import xml.etree.ElementTree as ET

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import fitz
from PIL import Image
from pypdf import PdfReader, PdfWriter
from google.colab import files
from tqdm.auto import tqdm

print("✓ Environment Ready")

Installing required libraries...
✓ Environment Ready


### Imports ###

In [2]:
# Standard Library
import io
import json
import logging
import os
import shutil
import subprocess
import tempfile
import time
import xml.etree.ElementTree as ET
import zipfile

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

# Third-Party Libraries
import fitz                      # PyMuPDF
import PIL

from PIL import Image
from pypdf import PdfReader, PdfWriter
from tqdm.auto import tqdm

# Google Colab
from google.colab import files

### High-Optimization Helper Functions & Implementations ###

In [5]:
from pathlib import Path
import os
import json
import xml.etree.ElementTree as ET
import zipfile
import subprocess
import shutil
import io

# Try to import optional packages for advanced file compression
try:
    from PIL import Image
except ImportError:
    Image = None

try:
    from pypdf import PdfReader, PdfWriter
except ImportError:
    PdfReader, PdfWriter = None, None

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

# Try to import Google Colab specific download utility
try:
    from google.colab import files
except ImportError:
    files = None


# ==========================================================
# HIGH-OPTIMIZATION HELPER FUNCTIONS & IMPLEMENTATIONS
# ==========================================================

def file_size(path):
    """Returns the size of a file in MegaBytes (MB)."""
    return os.path.getsize(path) / (1024 * 1024)


def compress_image_data(image_bytes, ext, mode="standard"):
    """
    Advanced imaging pipeline to compress raw image bytes dynamically.
    Optimizes Huffman trees and downsamples pixels for maximum byte savings.
    """
    if Image is None:
        return image_bytes

    try:
        img = Image.open(io.BytesIO(image_bytes))
        img_format = img.format or ext.replace(".", "").upper()
        if img_format == "MPO":
            img_format = "JPEG"

        # Max Optimization Mode: Resize large assets and apply tight quality thresholds
        if mode == "aggressive":
            max_dimension = 1400  # Capped for web/screen delivery
            if max(img.size) > max_dimension:
                img.thumbnail((max_dimension, max_dimension), Image.Resampling.LANCZOS)
            quality = 50
        else:
            quality = 65

        # Format-specific deep optimizations
        if img.mode in ('RGBA', 'LA', 'P') and img_format in ('JPEG', 'JPG'):
            img = img.convert('RGB')

        out_buffer = io.BytesIO()
        if img_format in ('JPEG', 'JPG'):
            # optimize=True runs an extra pass to find the absolute smallest Huffman tree
            img.save(out_buffer, format="JPEG", quality=quality, optimize=True, progressive=True)
        elif img_format == 'PNG':
            if mode == "aggressive":
                # Quantize PNG to an adaptive 8-bit palette (Drastic size drop)
                img = img.convert('P', palette=Image.Palette.ADAPTIVE, colors=256)
                img.save(out_buffer, format="PNG", optimize=True)
            else:
                img.save(out_buffer, format="PNG", optimize=True, compress_level=9)
        else:
            img.save(out_buffer, format=img_format)

        return out_buffer.getvalue()
    except Exception:
        return image_bytes


def compress_image(path, output_path, mode="standard"):
    """Compresses standalone image files by optimizing compression profiles."""
    try:
        with open(path, 'rb') as f:
            original_data = f.read()
        compressed_data = compress_image_data(original_data, path.suffix, mode=mode)
        with open(output_path, 'wb') as f:
            f.write(compressed_data)
    except Exception as e:
        print(f"  [Warning] Image compression failed: {e}")
        shutil.copy(path, output_path)


def compress_pdf(path, output_path, mode="standard"):
    """
    Multi-Strategy PDF Optimizer.
    Runs multiple optimization strategies and dynamically selects the absolute smallest output.
    """
    strategies = {}

    # Define temp file paths for evaluating different compression passes
    temp_gs = Path(f"{output_path}.gs.tmp")
    temp_fitz = Path(f"{output_path}.fitz.tmp")
    temp_pypdf = Path(f"{output_path}.pypdf.tmp")

    # Strategy 1: System Ghostscript Engine (Cleans vectors, flattens fonts, downsamples images)
    if shutil.which("gs"):
        pdf_settings = "/screen" if mode == "aggressive" else "/ebook"
        target_dpi = "72" if mode == "aggressive" else "150"
        try:
            cmd = [
                "gs", "-sDEVICE=pdfwrite", "-dCompatibilityLevel=1.4",
                f"-dPDFSETTINGS={pdf_settings}",
                "-dDownsampleColorImages=true", f"-dColorImageResolution={target_dpi}",
                "-dDownsampleGrayImages=true", f"-dGrayImageResolution={target_dpi}",
                "-dDownsampleMonoImages=true", f"-dMonoImageResolution={target_dpi}",
                "-dColorImageDownsampleType=/Bicubic",
                "-dGrayImageDownsampleType=/Bicubic",
                "-dNOPAUSE", "-dQUIET", "-dBATCH",
                f"-sOutputFile={temp_gs}", str(path)
            ]
            subprocess.run(cmd, check=True)
            if temp_gs.exists() and temp_gs.stat().st_size > 0:
                strategies["Ghostscript"] = temp_gs
        except Exception as e:
            if temp_gs.exists():
                temp_gs.unlink()

    # Strategy 2: PyMuPDF Extraction & Garbage Collection Pass
    if fitz is not None:
        try:
            doc = fitz.open(path)
            if mode == "aggressive" and Image is not None:
                for page in doc:
                    for img_info in page.get_images():
                        xref = img_info[0]
                        try:
                            base_image = doc.extract_image(xref)
                            if base_image:
                                compressed_img = compress_image_data(
                                    base_image["image"], f".{base_image['ext']}", mode="aggressive"
                                )
                                doc.replace_image(xref, data=compressed_img)
                        except Exception:
                            continue
            doc.save(
                str(temp_fitz),
                garbage=4,       # Deep search for unreferenced and duplicate streams
                deflate=True,    # Force deflate encryption streams
                clean=True       # Rectify damaged coordinates
            )
            doc.close()
            if temp_fitz.exists() and temp_fitz.stat().st_size > 0:
                strategies["PyMuPDF"] = temp_fitz
        except Exception as e:
            if temp_fitz.exists():
                temp_fitz.unlink()

    # Strategy 3: Basic Content Stream compression (pypdf)
    if PdfReader is not None:
        try:
            reader, writer = PdfReader(path), PdfWriter()
            for page in reader.pages:
                page.compress_content_streams()
                writer.add_page(page)
            writer.add_metadata({})
            with open(temp_pypdf, "wb") as f:
                writer.write(f)
            if temp_pypdf.exists() and temp_pypdf.stat().st_size > 0:
                strategies["pypdf"] = temp_pypdf
        except Exception:
            if temp_pypdf.exists():
                temp_pypdf.unlink()

    # Determine the winner (the file with the absolute lowest size)
    winner_name = None
    winner_path = None
    smallest_size = float('inf')

    for name, tmp_file in strategies.items():
        sz = tmp_file.stat().st_size
        if sz < smallest_size:
            smallest_size = sz
            winner_name = name
            winner_path = tmp_file

    if winner_path and winner_path.exists():
        print(f"  [PDF Selection] Winner Strategy: {winner_name} ({smallest_size / (1024*1024):.2f} MB)")
        shutil.move(str(winner_path), str(output_path))
    else:
        # Final fallback
        shutil.copy(path, output_path)
        print("\n  [Action Required] No active PDF compression engines succeeded!")
        print("  Please install system dependencies to enable high compression ratios.")

    # Cleanup leftover strategy files
    for tmp_file in [temp_gs, temp_fitz, temp_pypdf]:
        if tmp_file.exists() and tmp_file != winner_path:
            try:
                tmp_file.unlink()
            except Exception:
                pass


def _recompress_zip_structure(path, output_path, mode="standard"):
    """
    Unpacks OpenXML standard structures, compresses underlying raw files,
    strips redundant document metadata, and recompresses using Level 9 DEFLATE.
    """
    try:
        with zipfile.ZipFile(path, 'r') as yin:
            with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as yout:
                for item in yin.infolist():
                    data = yin.read(item.filename)

                    # Target image files inside docx/pptx/xlsx packages
                    if mode == "aggressive" and Image is not None:
                        filename_lower = item.filename.lower()
                        # Checks both media/ directories and any raw image suffixes for robustness
                        is_image = any(filename_lower.endswith(e) for e in [".png", ".jpg", ".jpeg"])
                        if is_image:
                            suffix = Path(item.filename).suffix
                            data = compress_image_data(data, suffix, mode="aggressive")

                    yout.writestr(item, data)
    except Exception as e:
        shutil.copy(path, output_path)
        print(f"  [Warning] OpenXML extraction pipeline error: {e}")


def compress_docx(path, output_path, mode="standard"):
    _recompress_zip_structure(path, output_path, mode=mode)


def compress_excel(path, output_path, mode="standard"):
    _recompress_zip_structure(path, output_path, mode=mode)


def compress_pptx(path, output_path, mode="standard"):
    _recompress_zip_structure(path, output_path, mode=mode)


# ==========================================================
# UNIVERSAL FILE OPTIMIZER
# ==========================================================

def compress(path, mode="standard"):
    """
    Optimizes a wide variety of file formats.

    Parameters:
      path (str or Path): Path to the source file.
      mode (str): Compression level.
                  - "standard": Safely optimize structure & streams (default).
                  - "aggressive": Rescale, downsample, and heavily compress layout & media assets.
    """
    path = Path(path)
    ext = path.suffix.lower()
    mode = mode.lower()

    output = path.with_name(f"{path.stem}_compressed{ext}")

    print("=" * 70)
    print(f"Processing      : {path.name}")
    print(f"Mode            : {mode.upper()}")

    if not path.exists():
        print(f"Error: Source file '{path}' does not exist!")
        return

    original = file_size(path)
    print(f"Original Size   : {original:.2f} MB")

    try:
        # =====================================================
        # Automatically Register Available Optimizers
        # =====================================================
        optimizers = {}

        if "compress_pdf" in globals():
            optimizers[".pdf"] = compress_pdf

        if "compress_docx" in globals():
            optimizers[".docx"] = compress_docx

        if "compress_excel" in globals():
            optimizers[".xlsx"] = compress_excel

        if "compress_pptx" in globals():
            optimizers[".pptx"] = compress_pptx

        if "compress_image" in globals():
            for image_ext in [
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp",
                ".gif",
                ".tif",
                ".tiff"
            ]:
                optimizers[image_ext] = compress_image

        # =====================================================
        # Binary Files & Dynamic Wrappers to support mode arg
        # =====================================================
        if ext in optimizers:
            optimizers[ext](path, output, mode=mode)

        # =====================================================
        # CSV
        # =====================================================
        elif ext == ".csv":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                rows = [
                    line.rstrip()
                    for line in f
                    if line.strip()
                ]

            with open(output, "w", encoding="utf-8") as f:
                f.write("\n".join(rows))

        # =====================================================
        # TXT
        # =====================================================
        elif ext == ".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()

            text = "\n".join(
                line.rstrip()
                for line in text.splitlines()
                if line.strip()
            )

            with open(output, "w", encoding="utf-8") as f:
                f.write(text)

        # =====================================================
        # JSON
        # =====================================================
        elif ext == ".json":
            with open(path, encoding="utf-8") as f:
                obj = json.load(f)

            with open(output, "w", encoding="utf-8") as f:
                json.dump(
                    obj,
                    f,
                    separators=(",", ":"),
                    ensure_ascii=False
                )

        # =====================================================
        # XML
        # =====================================================
        elif ext == ".xml":
            tree = ET.parse(path)
            # Minimize xml by removing unnecessary spaces on write
            tree.write(
                output,
                encoding="utf-8",
                xml_declaration=True
            )

        # =====================================================
        # HTML / CSS / JS
        # =====================================================
        elif ext in [".html", ".htm", ".css", ".js"]:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()

            text = " ".join(text.split())

            with open(output, "w", encoding="utf-8") as f:
                f.write(text)

        # =====================================================
        # Unsupported
        # =====================================================
        else:
            print(f"\n⚠ {ext} format is currently unsupported.")
            return

        # =====================================================
        # Compare File Sizes
        # =====================================================
        compressed = file_size(output)

        if compressed >= original:
            print("\n✓ File is already optimized.")
            print(f"Original : {original:.2f} MB")
            print(f"New File : {compressed:.2f} MB")

            try:
                output.unlink()
            except:
                pass

            print("Original file retained.")
            return

        reduction = (
            (original - compressed)
            / original
            * 100
        )

        print(f"Optimized Size : {compressed:.2f} MB")
        print(f"Reduction      : {reduction:.2f}%")
        print(f"Output File    : {output.name}")

        if files is not None:
            files.download(str(output))
        else:
            print(f"Saved locally: {output.resolve()}")

    except Exception as e:
        print("\nOptimization Failed")
        print(e)

    print("=" * 70)

### Import File for Compression ###

In [6]:
print("Upload one or more files")
uploaded = files.upload()

for filename in uploaded.keys():
    compress(filename)


Upload one or more files


Saving ATX ON A PAGE FA24 - Rupeshkumar Bomali.pdf to ATX ON A PAGE FA24 - Rupeshkumar Bomali.pdf
Processing      : ATX ON A PAGE FA24 - Rupeshkumar Bomali.pdf
Mode            : STANDARD
Original Size   : 144.92 MB
  [PDF Selection] Winner Strategy: PyMuPDF (3.87 MB)
Optimized Size : 3.87 MB
Reduction      : 97.33%
Output File    : ATX ON A PAGE FA24 - Rupeshkumar Bomali_compressed.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>